# Guardrails & Safety

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangChain roadmap.

You will learn how to use PIIMiddleware and custom middleware to protect agents from processing sensitive data.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. PIIMiddleware — Redact Strategy

Automatically detect and redact PII from user messages.

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langgraph.prebuilt.middleware import PIIMiddleware
from langchain_core.messages import HumanMessage

model = init_chat_model("gpt-4o-mini", model_provider="openai")

agent = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
    middleware=[PIIMiddleware(strategy="redact")],
)

result = agent.invoke({
    "messages": [HumanMessage(content="My email is alice@example.com and my card is 4111-1111-1111-1111")]
})
print(result["messages"][-1].content)

## 4. Mask Strategy

Partially hide PII values instead of fully redacting.

In [ ]:
agent_mask = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
    middleware=[PIIMiddleware(strategy="mask")],
)

result = agent_mask.invoke({
    "messages": [HumanMessage(content="Contact me at bob@company.com or call 555-123-4567")]
})
print(result["messages"][-1].content)

## 5. Hash Strategy

In [ ]:
agent_hash = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
    middleware=[PIIMiddleware(strategy="hash")],
)

result = agent_hash.invoke({
    "messages": [HumanMessage(content="My IP is 192.168.1.100")]
})
print(result["messages"][-1].content)

## 6. Block Strategy

Reject messages entirely if PII is detected.

In [ ]:
agent_block = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
    middleware=[PIIMiddleware(strategy="block")],
)

result = agent_block.invoke({
    "messages": [HumanMessage(content="My SSN is 123-45-6789")]
})
print(result["messages"][-1].content)

## 7. Custom Middleware

Build custom middleware with `before_agent` and `after_agent` hooks.

In [ ]:
from langgraph.prebuilt.middleware import AgentMiddleware

class ContentFilterMiddleware(AgentMiddleware):
    def __init__(self, blocked_words=None):
        self.blocked_words = blocked_words or []

    def before_agent(self, state):
        last_message = state["messages"][-1].content
        for word in self.blocked_words:
            if word.lower() in last_message.lower():
                state["messages"][-1].content = "[MESSAGE BLOCKED: prohibited content]"
                break
        return state

    def after_agent(self, state):
        last_message = state["messages"][-1].content
        for word in self.blocked_words:
            if word.lower() in last_message.lower():
                state["messages"][-1].content = "I can't help with that topic."
                break
        return state

content_filter = ContentFilterMiddleware(blocked_words=["hack", "exploit"])

agent_filtered = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
    middleware=[content_filter],
)

result = agent_filtered.invoke({
    "messages": [HumanMessage(content="How do I hack a website?")]
})
print(result["messages"][-1].content)

## 8. Layered Protection

Combine multiple middleware for defense in depth.

In [ ]:
pii_filter = PIIMiddleware(strategy="redact")
content_filter = ContentFilterMiddleware(blocked_words=["hack", "exploit"])

agent_layered = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
    middleware=[pii_filter, content_filter],
)

result = agent_layered.invoke({
    "messages": [HumanMessage(content="My email is alice@example.com, can you help me with this?")]
})
print(result["messages"][-1].content)

## Summary

- `PIIMiddleware` detects emails, credit cards, IPs, phone numbers, and SSNs
- Strategies: `redact`, `mask`, `hash`, `block`
- Custom middleware uses `before_agent` and `after_agent` hooks
- Layer multiple middleware for comprehensive protection
- Middleware runs in order, each processing the result of the previous